# The burnt marker: an orientation-proof reacting closure

A reacting network labels each edge **frozen** (unburnt reactants) or **equilibrium** (burnt products).
Getting that label from the *drawn* edge arrows is fragile: an edge may legitimately carry a negative mass
flow, so a flame edge drawn pointing the wrong way is silently mislabeled.

The fix is a transported **burnt marker** scalar `b`.
Every reacting edge runs a single `EQ_MARKER` closure that blends the frozen and equilibrium states by a
smooth gate of `b`; the flame stamps `b = 1` on whatever edge the flow actually leaves it by (the marker
rides the *signed* mass flow), so the split is correct no matter how the edges were drawn.
The topology flood-fill survives only as the marker's initial guess -- the transport self-corrects it.

In [ ]:
import os
import tempfile

import numpy as np
import plotly.graph_objects as go

import nefes
from nefes.thermo import ThermoInp, Thermo, EQ_FROZEN, EQ_KERNEL, EQ_MARKER
from nefes.thermo.edge_state import MARKER_GATE_WIDTH
from nefes.chem import resolve_composition, enthalpy_mass
from nefes.elements import catalog as cat
from nefes.assembly.smooth import marker_gate
from nefes.plotting import use_nefes_theme, COLORWAY

use_nefes_theme()

lib = ThermoInp().library(["H2", "O2", "N2", "H2O", "OH", "H", "O", "HO2"])
gas = Thermo(lib)
AIR = {"O2": 0.21, "N2": 0.79}
Y_air, _ = resolve_composition(lib, AIR, basis="mole")
H_AIR = enthalpy_mass(lib, Y_air, 300.0)  # absolute-enthalpy datum for the air feed

## The smooth blend gate

The closure blends `(1 - g)` of the frozen state with `g` of the equilibrium state, where `g = marker_gate(b)`.
The gate is normalized so `g(0) = 0` and `g(1) = 1` *exactly* -- a fresh edge is pure frozen, a burnt edge pure
equilibrium -- while staying smooth (and complex-step-safe) in between, so the coupled Newton solve sees no
branch.
The marker is bimodal at convergence, so the blend is only ever active during the iteration.

In [ ]:
b = np.linspace(-0.3, 1.3, 400)
g = np.array([marker_gate(float(x), MARKER_GATE_WIDTH) for x in b])
fig = go.Figure()
fig.add_trace(go.Scatter(x=b, y=g, mode="lines", line=dict(color=COLORWAY[4], width=3),
                         name="g = marker_gate(b)"))
fig.add_trace(go.Scatter(x=[0, 1], y=[0, 1], mode="markers",
                         marker=dict(size=10, color=COLORWAY[0]), name="converged b in {0, 1}"))
fig.update_layout(width=620, height=380,
                  title="Blend gate: 0 = frozen (unburnt), 1 = equilibrium (burnt)",
                  xaxis_title="burnt marker  b", yaxis_title="equilibrium weight  g")
fig.show()

## A marker-gated combustor

Air enters, a fuel stream is injected, a flame sits mid-duct, products leave.
We build it with the plain `Network` API and **no** per-edge closure override, so it is marker-gated
automatically: every edge is `EQ_MARKER` and one extra `burnt` scalar is transported.

![Network topology](burnt_marker_topology.png)

In [ ]:
def combustor(edges, edge_models=None):
    """Air feed, H2 injection, a mid-duct flame, and a mass-flow outlet.

    With ``edge_models=None`` every edge is marker-gated (``EQ_MARKER``); pass an explicit
    per-edge list to pin a hard frozen/equilibrium closure instead.
    """
    nodes = [
        cat.total_pressure_inlet(1.2e5, 300.0, composition=AIR, basis="mole", name="air"),
        cat.duct(0.4),
        cat.mass_source(0.006, 300.0, composition={"H2": 1.0}, basis="mole", name="H2"),
        cat.duct(0.4),
        cat.equilibrium_flame(name="flame"),
        cat.duct(0.5),
        cat.mass_flow_outlet(0.406, name="out"),
    ]
    return nefes.Network(nefes.equilibrium(lib),
                         nodes=nodes, edges=edges, edge_models=edge_models)


# edge 4 is the post-flame (burnt) edge; the rest are the unburnt approach
edges = [(0, 1, 0.01), (1, 2, 0.01), (2, 3, 0.01), (3, 4, 0.01), (4, 5, 0.01), (5, 6, 0.01)]
net = combustor(edges)

# With no per-edge override every edge is marker-gated and one burnt scalar is transported;
# the flood-fill only supplies the marker's initial guess.
prob = net.compile()
print("per-edge model :", ["EQ_MARKER" if m == EQ_MARKER else int(m) for m in prob.edge_model])
print("marker seed    :", prob.marker_seed.tolist(), " <- the flood-fill, only an initial guess")

sol = net.solve()
print("converged      :", sol.converged, " in", sol.iterations, "iterations")

In [ ]:
E = len(sol.field("T"))
b_field = [sol.marker(e) for e in range(E)]
T = sol.field("T")
fig = go.Figure()
fig.add_trace(go.Bar(x=list(range(E)), y=b_field, marker_color=COLORWAY[0], name="burnt marker b",
                     yaxis="y1"))
fig.add_trace(go.Scatter(x=list(range(E)), y=T, mode="lines+markers", line=dict(color=COLORWAY[4], width=3),
                         name="static T [K]", yaxis="y2"))
fig.update_layout(width=720, height=400,
                  title="Marker = 1 exactly where the gas is burnt; T jumps with it",
                  xaxis_title="edge",
                  yaxis=dict(title="burnt marker", range=[0, 1.05]),
                  yaxis2=dict(title="T [K]", overlaying="y", side="right", showgrid=False))
fig.show()

## Orientation robustness: the seed is only a guess

The flood-fill that seeds the marker follows the *drawn* arrows, so a backward-drawn flame seeds it wrong.
It does not matter: the marker transport rides the signed mass flow, so it converges to the **same** field
regardless of its starting guess.
Below we scramble the initial marker three different (wrong) ways -- overwriting only the marker row of the
converged state -- and recover the same answer each time.

In [ ]:
mr = prob.marker_row  # the state row carrying the transported burnt marker
scrambles = {
    "flood-fill seed": prob.marker_seed.tolist(),
    "all fresh (0)":   [0, 0, 0, 0, 0, 0],
    "inverted":        [1, 1, 1, 1, 0, 0],
    "all burnt (1)":   [1, 1, 1, 1, 1, 1],
}
fig = go.Figure()
for name, seed in scrambles.items():
    x0 = sol.x.copy()
    x0[mr, :] = seed  # scramble only the marker row, keep the converged flow
    r = combustor(edges).solve(x0=x0)
    conv = [round(r.marker(e), 4) for e in range(E)]
    print(f"{name:16s} seed={seed} -> converged marker={conv}  (conv={r.converged})")
    fig.add_trace(go.Scatter(x=list(range(E)), y=conv, mode="lines+markers", name=name))
fig.update_layout(width=720, height=380,
                  title="Every scrambled seed converges to the same marker field",
                  xaxis_title="edge", yaxis_title="converged burnt marker")
fig.show()

## Post-processing: marker, species, molar mass, cp

Everything the closure computes is surfaced per edge after the solve (and goes into the YAML output by
default).
The species readout follows the marker: an unburnt edge reports its frozen reactants, a burnt edge its
equilibrium products.

In [ ]:
for e in (3, 4):  # the flame-approach (fresh) edge and the flame-out (burnt) edge
    tag = "BURNT " if sol.marker(e) > 0.5 else "fresh "
    top = sorted(sol.species(e, basis="mole").items(), key=lambda kv: -kv[1])[:4]
    sp = ", ".join(f"{n} {x:.3f}" for n, x in top)
    st = sol.edge(e)
    print(f"edge {e} [{tag}] b={sol.marker(e):.3f}  T={st['T']:7.1f} K  "
          f"W={st['W'] * 1e3:5.2f} g/mol  cp={st['cp']:6.1f} J/kgK  | {sp}")

## The marker is acoustically passive

At convergence the marker is saturated (`b in {0, 1}`), so the gate is flat there and the marker decouples
from the acoustics -- it is just a convected scalar.
The acoustic transfer matrix across the flame is therefore identical to the one from an explicit hard
`EQ_FROZEN` / `EQ_KERNEL` closure (no marker).

In [ ]:
freqs = np.linspace(50.0, 800.0, 60)

# the same geometry with an explicit hard closure (frozen approach, equilibrium downstream)
hard_models = [EQ_FROZEN, EQ_FROZEN, EQ_FROZEN, EQ_FROZEN, EQ_KERNEL, EQ_KERNEL]
sol_hard = combustor(edges, edge_models=hard_models).solve()

rm = sol.perturbation_response(freqs, excite=("acoustic", "entropy"))
rh = sol_hard.perturbation_response(freqs, excite=("acoustic", "entropy"))
Tm = rm.transfer_matrix(0, 4)  # inlet edge -> burnt edge, across the flame
Th = rh.transfer_matrix(0, 4)

fig = go.Figure()
fig.add_trace(go.Scatter(x=freqs, y=np.abs(Tm[:, 0, 0]), mode="lines", line=dict(color=COLORWAY[4], width=4),
                         opacity=0.5, name="marker-gated  |T(f→f)|"))
fig.add_trace(go.Scatter(x=freqs, y=np.abs(Th[:, 0, 0]), mode="lines", line=dict(color=COLORWAY[0], dash="dot"),
                         name="hard closure  |T(f→f)|"))
fig.update_layout(width=720, height=380,
                  title="Marker-gated vs hard-closure acoustic transfer across the flame (overlaid)",
                  xaxis_title="frequency [Hz]", yaxis_title="|T| (downstream acoustic)")
fig.show()
print("max |T_marker - T_hard| over the sweep:", float(np.max(np.abs(Tm - Th))))

## User-settable inflow marker: feeding already-burnt gas

An inlet (or source) normally enters *fresh* (`marker = 0`, the default).
Setting `marker = 1` injects **already-burnt** gas -- e.g. exhaust-gas recirculation as a feed -- which forces
the equilibrium closure on the inflow edge itself, so a premixed feed burns right at the inlet instead of
staying frozen until the downstream flame.
(The marker only applies on a marker-gated network -- a reacting net with an equilibrium flame and no explicit
per-edge closure -- so a non-zero value elsewhere is rejected at build time.)

In [ ]:
PREMIX = {"H2": 0.296, "O2": 0.148, "N2": 0.556}  # stoichiometric H2-air (by mole)
Y_premix, _ = resolve_composition(lib, PREMIX, basis="mole")
H_PREMIX = enthalpy_mass(lib, Y_premix, 300.0)  # absolute-enthalpy datum for the premix feed


def premixed(marker):
    net = nefes.Network(
        nefes.equilibrium(lib),
        nodes=[
            cat.total_pressure_inlet(1.2e5, 300.0, composition=PREMIX, basis="mole", marker=marker, name="feed"),
            cat.equilibrium_flame(name="flame"),
            cat.pressure_outlet(1.0e5, 300.0, composition=PREMIX, basis="mole", name="out"),
        ],
        edges=[(0, 1, 0.01), (1, 2, 0.01)],
    )
    return net.solve()


for m in (0.0, 1.0):
    s = premixed(m)
    T = s.field("T")
    tag = "fresh feed: frozen until the flame" if m == 0 else "burnt feed: equilibrium at the inlet"
    print(f"marker={m:.0f}:  inflow-edge T = {T[0]:7.1f} K,  outlet T = {T[1]:7.1f} K   [{tag}]")

## Export for Nemo

The marker-gated combustor `net` and its converged mean flow `sol` are already in scope.
Save either to a UI-readable YAML -- `sol.to_yaml(path)` embeds the mean-flow result fields,
`net.to_yaml(path)` writes the topology only -- then open the file in Nemo.

In [ ]:
_out = os.path.join(tempfile.mkdtemp(), "burnt_marker.yaml")
sol.to_yaml(_out)  # embeds the mean-flow results; use net.to_yaml(_out) for topology only
print("saved case:", _out)